### Neural Network

In [2]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

### Tensors

[Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) - ტენსორი არის N-განზომილებიანი რიცხვების მასივი.

მანქანურ სწავლებაში (ML) ტენსორები ინახავენ მონაცემებს (inputs, labels) და მოდელის პარამეტრებს (weights, biases). 

შეგიძლიათ შექმნათ, ინდექსირება გაუკეთოთ, გადააკეთოთ ფორმა (reshape), გაიჭრათ (slice), გააგზავნოთ ბროდკასტირება (broadcast) და ჩაატაროთ არითმეტიკული/ალგებრული ოპერაციები ტენსორებზე ყველა სამივე ბიბლიოთეკაში (NumPy, PyTorch, TensorFlow).

In [14]:
# Tensors in NumPy arrays

# A scalar (0D tensor)
scalar = np.array(42)       # shape: ()

# A vector (1D tensor)
vector = np.array([1, 2, 3])  # shape: (3,)

# A matrix (2D tensor)
matrix = np.array([[1, 2, 3], [4, 5, 6]])  # shape: (2, 3)

# A 3D tensor
tensor3d = np.zeros((2, 3, 4))  # shape: (2, 3, 4), all zeros

print("scalar.shape:", scalar.shape)        # → ()
print("vector.shape:", vector.shape)        # → (3,)
print("matrix.shape:", matrix.shape)        # → (2, 3)
print("tensor3d.shape:", tensor3d.shape)    # → (2, 3, 4)

scalar.shape: ()
vector.shape: (3,)
matrix.shape: (2, 3)
tensor3d.shape: (2, 3, 4)


In [15]:
# Tensors in PyTorch

# Scalar (0D)
scalar = torch.tensor(3.14)     # dtype=torch.float32 by default

# Vector (1D)
vector = torch.tensor([1.0, 2.0, 3.0])

# Matrix (2D)
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

# 3D Tensor, filled with zeros
tensor3d = torch.zeros((2, 3, 4))  # dtype=torch.float32

# Tensor with random values
rand_tensor = torch.randn((5, 5))  # standard normal distribution

print("tensor3d.shape:", tensor3d.shape)  # → torch.Size([2, 3, 4])

tensor3d.shape: torch.Size([2, 3, 4])


In [17]:
# Moving Tensors to GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

x_cpu = torch.randn((2, 2))
x_gpu = x_cpu.to(device)      # copies tensor to GPU if available
print(x_gpu.device)           # → cuda:0 (if GPU) or cpu

Using device: cpu
cpu


In [10]:
# Basic Operations
a = torch.tensor([[1.0, 2.0], 
                  [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0],
                  [30.0, 40.0]])

# Addition
c = a + b    # elementwise, shape (2,2)

# Matrix multiplication (dot product)
d = torch.matmul(a, b)  # same as a @ b

# Elementwise multiplication
e = a * b

# Reshaping
f = a.view(1, 4)       # shape (1, 4)
g = a.view(-1, 1)      # shape (4, 1)

print("c:", c)
print("d:", d)
print("e:", e)
print("f.shape:", f)
print("g.shape:", g.shape)

c: tensor([[11., 22.],
        [33., 44.]])
d: tensor([[ 70., 100.],
        [150., 220.]])
e: tensor([[ 10.,  40.],
        [ 90., 160.]])
f.shape: tensor([[1., 2., 3., 4.]])
g.shape: torch.Size([4, 1])


#### Gradient
Gradient (∇s) is the vector of partial derivatives of s with respect to each component of x. 

მაგალითად, 

$$
y_i = 3x_i^2 + 2x_i,\quad s = \sum_i y_i = \sum_i \bigl(3x_i^2 + 2x_i\bigr).
$$

ანუ თითოეული $x_i$ კომპონენტისთვის,

$$
\frac{\partial s}{\partial x_i}
= \frac{\partial}{\partial x_i}\bigl(3x_i^2 + 2x_i\bigr)
= 6x_i + 2.
$$

როცა $x = [2.0,\,3.0]$:

* პირველი კომპონენტისთვის ($x_1 = 2$):
  $\displaystyle \frac{\partial s}{\partial x_1} = 6\cdot 2 + 2 = 14.$
* მეორე კომპონენტისთვის ($x_2 = 3$):
  $\displaystyle \frac{\partial s}{\partial x_2} = 6\cdot 3 + 2 = 20.$

გრადიენტის ვექტორი: $[14,\;20]$.

In [20]:
# Create a tensor with requires_grad=True so PyTorch tracks operations
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Define a simple function: y = 3*x^2 + 2*x
y = 3 * x**2 + 2 * x   

# Suppose we want the gradient of sum(y) wrt x
s = y.sum()              # scalar
s.backward()             # compute gradients

# Now x.grad holds d(s)/d(x) = [d/dx for each element]
print("x:", x)           # [2.0, 3.0]
print("y:", y)           # [ 3*(2)^2+2*2=16, 3*(3)^2+2*3=33 ]
print("x.grad:", x.grad) # [d/dx (16)= (6*2 + 2)=14, d/dx(33)= (6*3 + 2)=20]

x: tensor([2., 3.], requires_grad=True)
y: tensor([16., 33.], grad_fn=<AddBackward0>)
x.grad: tensor([14., 20.])


#### Loss

In [23]:
# Suppose these are model outputs and true targets:
y_pred = torch.tensor([2.0, 0.5, 3.7])   # shape (3,)
y_true = torch.tensor([1.5, 1.0, 4.0])   # shape (3,)

# Compute MSE loss (averaged over 3 examples)
mse_loss = F.mse_loss(y_pred, y_true, reduction="mean")
print("MSE loss:", mse_loss.item())
# → ((2.0−1.5)^2 + (0.5−1.0)^2 + (3.7−4.0)^2) / 3 
#   = (0.25 + 0.25 + 0.09) / 3 = 0.1967…

MSE loss: 0.1966666579246521


In [217]:
# Loss ფუნქცია იღებს (output, target) ითვლის რამდენად დაშორებულია ისინი ერთმანეთისგან.
criterion = nn.MSELoss()
loss = criterion(y_pred, y_true)
loss

tensor(1.3919, grad_fn=<MseLossBackward0>)

1. Forward pass
   tensor ოპერაციების შედეგად ვიღებთ გარკვეულ output-ს, იგება computation graph

   ```python
   y = 3 * x**2 + 2 * x
   loss = y.sum()
   ```
2. Triggering .backward()
   როცა გავუშვებთ

   ```python
   loss.backward()
   ```
   PyTorch იწყებს backward walks computation graph-ის წვეროებზე და chain rule-ს იყენებს, მან იცის “how the output changes if its inputs change.”

3. როგორ მუშაობს Chain Rule

   * მაგალითად, თუ გვაქვს $y = 3 x^2 + 2 x$, მაშინ ლოკალური წარმოებული არის $\partial y/\partial x = 6x + 2$.
   * რამდენიმე ნიმუშის შემთხვევაში: $loss = \sum_i y_i$, მაშინ გვაქვს $\frac{\partial loss}{\partial x_i}
= \frac{\partial loss}{\partial y_i}
\cdot \frac{\partial y_i}{\partial x_i}
= 1 \cdot (6x_i + 2). $`loss.backward()` ითვალისწინებს, რომ $\partial loss/\partial y_i = 1.$

   * მაგალითად, თუ $x = [2,3]$, მაშინ $\partial loss/\partial x_1 = 6·2 + 2 = 14$, $\partial loss/\partial x_2 = 6·3 + 2 = 20$.

4. გრადიენტების დაგროვება
   * თუ tensor-ს აქვს `requires_grad=True`, მის `tensor.grad`-ში ავტომატურად ჩაიწერება $\partial loss/\partial tensor$.
   * w.grad  # now holds ∂L/∂w
     
5. მონაცემთა განლაგება GPU/CPU-ზე
   თუ tensor CPU-ზეა, გრადიენტიც CPU-ზე დაითვლება; GPU-ზეა თუ tensor, gradient-იც GPU-ზე გამოითვლება.

6. მაგალითი

   ```python
   x = torch.tensor([2.0, 3.0], requires_grad=True)
   y = 3 * x**2 + 2 * x       # y_i = 3·x_i^2 + 2·x_i
   loss = y.sum()            # loss = Σ y_i
   loss.backward()           # აქ PyTorch ავლენს chain rule-ს
   print(x.grad)             # [14, 20]
   ```

“if I nudge this tensor a tiny bit, how much will the loss change?” 

<img src="../../images/gd.jpg" width="500">

Batch gradient descent შეიძლება გაიჭედოს ლოკალურ მინიმუმებში, 
ან ძალიან ნელა პროგრესირებდეს ბრტყელ (flat) რეგიონებში.
Stochastic და mini-batch gradient descent ამატებენ ხმაურს (noise),
რაც ეხმარება ალგორითმს ლოკალური მინიმუმებიდან გამოსვლასა და პლატოების გადალახვაში. 

წრფივი რეგრესიის (linear regression) loss ფუნქციისათვის:

* არსებობს მხოლოდ ერთი გლობალური მინიმუმი
* ლოკალური მინიმუმები არ არსებობს
* პლატოები არ გვხვდება

| Optimizer                             | Typical models                         | How local minima / plateaus are handled          | Advantages                           | Disadvantages                                 |
| ------------------------------------- | -------------------------------------- | ------------------------------------------------ | ------------------------------------ | --------------------------------------------- |
| **Batch Gradient Descent**            | Linear Regression, convex models       | Not handled – deterministic path                 | Stable, exact gradient, simple       | Slow, stuck in local minima, poor scalability |
| **Stochastic Gradient Descent (SGD)** | Neural Networks, Logistic Regression   | Noise helps escape local minima and plateaus     | Fast, scalable, good generalization  | Noisy updates, unstable convergence           |
| **Mini-batch Gradient Descent**       | Neural Networks (standard)             | Controlled noise enables escape from bad regions | Efficient, stable, GPU-friendly      | Batch-size tuning required                    |
| **SGD + Momentum**                    | Deep Neural Networks                   | Velocity helps cross plateaus and shallow minima | Faster convergence, less oscillation | Extra hyperparameter (momentum)               |
| **Nesterov Momentum**                 | Deep Neural Networks                   | Look-ahead gradient prevents overshooting        | More stable than momentum            | More complex, similar gains                   |
| **Adagrad**                           | Sparse data (NLP, recommender systems) | Adaptive step sizes avoid flat regions           | Good for sparse features             | Learning rate decays too fast                 |
| **RMSProp**                           | RNNs, deep networks                    | Adaptive learning rates smooth plateaus          | Stable, fast convergence             | No theoretical convergence guarantee          |
| **Adam**                              | Most Neural Networks                   | Momentum + adaptive learning rates               | Fast, robust, little tuning          | Sometimes worse generalization                |


### Neural Network
ხელოვნური ნეირონული ქსელი არის ალგორითმი, რომელიც ცდილობს იმუშაოს ადამიანური ტვინის პრინციპით — იღებს ინფორმაციას, ამუშავებს და აბრუნებს გადაწყვეტილებას.

ნეირონული ქსელი 3 ნაწილისგან შედგება:
| ნაწილი                             | როლი                                            |
| ---------------------------------- | ----------------------------------------------- |
| Input Layer    | იღებს მონაცემს          |
| Hidden Layer | ამუშავებს მონაცემს, აკეთებს გათვლებს            |
| Output Layer| აბრუნებს შედეგს |

<img src="../../images/one_neuron.png" width="800">

| ტერმინი    | მნიშვნელობა                                  |
| ---------- | -------------------------------------------- |
| Input      | ის, რასაც ვაძლევთ მოდელს                     |
| Output     | შედეგი, რომელსაც მოდელი აბრუნებს             |
| Weight     | წონა, რომელიც ახდენს გავლენას შედეგზე |
| Bias       | დამატებითი რიცხვი შედეგის რეგულირებისთვის    |
| Activation | ფუნქცია, რომელიც წყვეტს, რა დაბრუნდეს    |
| Loss       | შეცდომა — რამდენად შორსაა მოდელის პროგნოზი ნამდვილი მნიშვნელობისგან    |
| Epoch      | რამდენჯერ გადაუვლის მოდელი მთელ მონაცემებს (რამდენჯერ გაიმეორებს)  |

<img src="../../images/nn.webp" width="1000">

Weights და bias არის ორი ძირითადი trainable პარამეტრი ყველა ნეირონში (neuron), რომლებიც ისე გვინდა სწავლების პროცესში შევცვალოთ, რომ მივაღწიოთ საუკეთესო პროგნოზს (prediction).

---

#### weight

* თუ $x$ არის ნეირონის input ვექტორი $x = [x_1, x_2, \dots, x_n]$, $x_i$-ს ყოველ კომპონენტს აქვს მისი შესაბამისი weight, $w_i$.
* წრფივი ფუნქციის სახით:

    $$
      z = \sum_{i=1}^{n} w_i \, x_i.
    $$
* weight მიუთითებს, თუ რამდენად მნიშვნელოვანია ან რამდენად გავლენიანი არის კონკრეტული $x_i$ პროგნოზის (prediction) მიღებაში.

---

#### bias

  * წრფივი ფუნქცია :

    $$
      z = \sum_{i=1}^{n} w_i \, x_i \;+\; b,
    $$

    სადაც $b$ არის bias.

  * თუ bias არ გვექნება და ყველა $x_i=0$, მაშინ $z$ ყოველთვის 0 იქნება, რაც შესაძლოა არ შეესაბამებოდეს რეალობას.

---

#### weights & bias

* ვთქვათ, ნეირონს აქვს შემდეგი შემოსული სიგნალი (input):

   $$
     x_1,\, x_2,\, \dots,\, x_n.
   $$
* თითოეულ $x_i$-ს შევუსაბამებთ $w_i$-ს და შევკრიბავთ:

   $$
     z_{\text{linear}} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n.
   $$
* ამ წრფივ ჯამს მივუმატებთ bias-ს $b$:

   $$
     z = z_{\text{linear}} + b.
   $$
* Activation (მაგალითად, ReLU, Sigmoid, ა.შ.)
  საბოლოო ნაბიჯია activation ფუნქცია, მიღებული $z$ გადაეცემა activation ფუნქციას როგორც არგუმენტი, რაც გვაძლევს საბოლოო output–ს $a = \sigma(z)$.
  - Relu (Rectified Linear Unit)

![alt text](https://yashuseth.files.wordpress.com/2018/02/relu-function.png?w=309&h=274)
- Tanh (Hyperbolic Tangent)

![alt text](http://mathworld.wolfram.com/images/interactive/TanhReal.gif)
- Sigmoid 

![alt text](https://miro.medium.com/max/970/1*Xu7B5y9gp0iL5ooBj7LtWw.png)

---

#### update weight-ები და bias

* Training-ის პროცესში ჩვენ გვაქვს loss function, რომელიც გვაძლევს, რამდენად დაშორებულია ნეირონის output რეალური target-ისგან.
* Gradient descent–ის ან სხვა ოპტიმიზაციის ალგორითმის საშუალებით, ვითვლით
  $\frac{\partial \text{loss}}{\partial w_i}$ და $\frac{\partial \text{loss}}{\partial b}$.

  * ეს არის ინდიკატორი იმისა, თუ რამდენად უნდა შევცვალოთ $w_i$-ი და $b$, რომ შევამციროთ error.

  $$
    w_i \leftarrow w_i - \eta \,\frac{\partial \text{loss}}{\partial w_i}, \quad
    b \leftarrow b - \eta \,\frac{\partial \text{loss}}{\partial b},
  $$

  სადაც $\eta$ არის learning rate.
* ასე რეგულირდება თითოეული weight და bias.

---

#### მაგალითი ერთ ნეირონით

ვქვიოთ, გვაქვს მარტივი პრობლემა: სწორი $y$ უნდა განისაზღვროს $x$-ის საფუძველზე როგორც $y = 2x + 1$. ამ შემთხვევაში,

* ნეირონის ფუნქცია:

  $$
    y_{\text{pred}} = w\,x + b.
  $$
* რეალური target: $y_{\text{true}} = 2x + 1$.

Initialization

* დავიწყოთ შემთხვევითი $w$ და $b$-ით, მაგალითად $w=0.5$, $b=0.0$.

Forward pass

* პირველ iteration-ზე, თუ $x=3$, ნეირონი გამოითვლის:

  $$
    y_{\text{pred}} = 0.5 \times 3 + 0 = 1.5.
  $$
* ხოლო $y_{\text{true}} = 2×3 +1 = 7.$

Loss

* Loss (MSE) იქნება:

  $$
    L = (y_{\text{pred}} - y_{\text{true}})^2 = (1.5 - 7)^2 = 30.25.
  $$

Backward და Gradient

* $\frac{\partial L}{\partial w} = 2 (y_{\text{pred}} - y_{\text{true}})\, x$.
  თუ $y_{\text{pred}} - y_{\text{true}} = 1.5 - 7 = -5.5$ და $x=3$, მაშინ:

  $$
    \frac{\partial L}{\partial w} = 2 \times (-5.5) \times 3 = -33.
  $$
* $\frac{\partial L}{\partial b} = 2 (y_{\text{pred}} - y_{\text{true}})$.

  $$
    \frac{\partial L}{\partial b} = 2 \times (-5.5) = -11.
  $$

Update

* ავიღოთ learning rate $\eta=0.1$.

  $$
    w \leftarrow 0.5 - 0.1 \times (-33) = 0.5 + 3.3 = 3.8,  
  $$

  $$
    b \leftarrow 0.0 - 0.1 \times (-11) = 1.1.
  $$
* ასე, bias და weight უფრო მიახლოვდება რეალურ 2 და 1 მნიშვნელობებს.
---

In [12]:
# 1) y_true = 2*x + 1
x = torch.tensor([3.0]).unsqueeze(1)  
y_true = 2 * x + 1                   

# 2) Initialize weight w და bias b, რომლებსაც უნდა ჰქონდეთ requires_grad=True
w = torch.randn(1, requires_grad=True)  # შემთხვევითი სკალარული წონა
b = torch.zeros(1, requires_grad=True)  # საწყისად 0-ზე bias

# 3) Forward pass: y_pred = w * x + b
y_pred = w * x + b  

# 4) MSE Loss: (1/N) * sum((y_pred - y_true)^2), N = 1
loss = ((y_pred - y_true) ** 2).mean()

# 5) Backward pass: გამოთვლა ∂loss/∂w და ∂loss/∂b
loss.backward()

# 6) Inspect gradients:
print(f"w (before update): {w.item():.4f},  ∂loss/∂w = {w.grad.item():.4f}")
print(f"b (before update): {b.item():.4f},  ∂loss/∂b = {b.grad.item():.4f}")

# 7) Gradient Descent – ერთი ნაბიჯი განახლებისთვის
learning_rate = 0.1
with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()

print(f"w (after 1 update): {w.item():.4f}")
print(f"b (after 1 update): {b.item():.4f}")

w (before update): -1.1041,  ∂loss/∂w = -61.8730
b (before update): 0.0000,  ∂loss/∂b = -20.6243
w (after 1 update): 5.0832
b (after 1 update): 2.0624


წონების შეცვლა:
The simplest update rule: Stochastic Gradient Descent (SGD):

```weight = weight - learning_rate * gradient```

შესაბამისი კოდი:

```
learning_rate = 0.01
for f in net.parameters():
    f.data.sub_(f.grad.data * learning_rate)
```
```torch.optim``` ამ მეთოდის იმპლემენტაციას აკეთებს.

In [17]:
# y_true = 2*x + 1
x = torch.tensor([3.0]).unsqueeze(1)   # shape (1,1)
y_true = 2 * x + 1                     # shape (1,1)

# “Net” with one weight and one bias
class SingleNeuron(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Parameter wraps the tensor so that .parameters() will include it
        self.w = nn.Parameter(torch.randn(1))   # random scalar weight
        self.b = nn.Parameter(torch.zeros(1))   # zero bias

    def forward(self, x):
        return self.w * x + self.b

net = SingleNeuron()

# Forward pass: y_pred = w * x + b
y_pred = net(x)  # calls forward(x)

# MSE Loss
loss_fn = nn.MSELoss()    # same as ((y_pred - y_true)**2).mean()
loss = loss_fn(y_pred, y_true)

# Set up an SGD optimizer over net.parameters()
optimizer = optim.SGD(net.parameters(), lr=0.01)

print(f"After random initialisation: w={net.w.item():.4f},  b={net.b.item():.4f}")

# Training for 5 epochs
for epoch in range(1, 5):
    # Forward pass
    y_pred = net(x)
    loss = loss_fn(y_pred, y_true)

    # Backward + update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress
    print(f"Epoch {epoch:>2}: loss={loss.item():.4f},  w={net.w.item():.4f},  b={net.b.item():.4f}")

After random initialisation: w=1.1845,  b=0.0000
Epoch  1: loss=11.8790,  w=1.3913,  b=0.0689
Epoch  2: loss=7.6026,  w=1.5567,  b=0.1241
Epoch  3: loss=4.8656,  w=1.6891,  b=0.1682
Epoch  4: loss=3.1140,  w=1.7949,  b=0.2035
